# 03 · XGBoost

Gradient boosting con crecimiento por niveles (profundidad fija), lo que lo hace algo distinto a LightGBM y útil para ensamblar.

Usa las features generadas por `00_features.ipynb`. Validación: 5 folds estratificados, los mismos para todos los modelos; las features que dependen de la etiqueta están recalculadas dentro de cada fold.

In [1]:
import os, time, numpy as np, pandas as pd, warnings
import features as F
warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)
y = F.load_y(); folds = F.cv_folds(y)
import xgboost as xgb

## Modelo

In [2]:
PARAMS = dict(learning_rate=0.03, n_estimators=1200, max_depth=9, min_child_weight=2, colsample_bytree=0.3, subsample=0.8,
              reg_lambda=3, tree_method="hist", enable_categorical=True, max_cat_to_onehot=64, n_jobs=6)
SEEDS = [42, 43, 44]   # se promedian varias semillas para reducir la varianza del modelo

def fit_predict(A, ya, B, return_model=False):
    models = [xgb.XGBClassifier(**PARAMS, random_state=s).fit(A, ya) for s in SEEDS]
    p = np.mean([m.predict_proba(B)[:, 1] for m in models], axis=0)
    return (p, models) if return_model else p

## Validación cruzada

In [3]:
oof = np.zeros(len(y)); t0 = time.time()
for f, (a, b) in enumerate(folds):
    A, B = F.load_fold(f)
    oof[b] = fit_predict(A, y[a], B)
    print(f"fold {f}: accuracy {((oof[b] > .5) == y[b]).mean():.4f}  ({time.time() - t0:.0f}s)")
print(f"\nAccuracy CV: {((oof > .5) == y).mean():.4f}")
np.save(f"{F.CACHE}/oof_xgboost.npy", oof)

fold 0: accuracy 0.9496  (115s)


fold 1: accuracy 0.9466  (231s)


fold 2: accuracy 0.9491  (347s)


fold 3: accuracy 0.9462  (469s)


fold 4: accuracy 0.9464  (679s)

Accuracy CV: 0.9476


## Modelo final (todo train) y predicción de test

In [4]:
A, B = F.load_full()
proba, model = fit_predict(A, y, B, return_model=True)
for s, m in zip(SEEDS, model):
    m.save_model(f"models/xgboost_seed{s}.json")
np.save(f"{F.CACHE}/test_xgboost.npy", proba)
F.save_submission(proba, "xgboost")["condition"].value_counts(normalize=True)

condition
new     0.526867
used    0.473133
Name: proportion, dtype: float64